# 09 — Advanced Machine Learning & Tree Ensembles

**Objective:** Benchmark non-linear tree-based and gradient boosting models:
- DecisionTreeRegressor
- RandomForestRegressor
- ExtraTreesRegressor
- HistGradientBoostingRegressor
- XGBoost (XGBRegressor)
- CatBoost (CatBoostRegressor)


In [1]:
import sys
import time
from pathlib import Path
import pandas as pd
import numpy as np

def find_root():
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / 'data' / 'raw' / 'gt_2011.csv').exists():
            return parent
    return p

ROOT = find_root()
sys.path.insert(0, str(ROOT))

from src.models.model_factory import create_model
from src.data.loader import FEATURES_A, TARGETS_A
from src.data.splitting import temporal_split
from src.evaluation.metrics import regression_metrics


## 1. Training & Evaluating Advanced Models


In [2]:
df = pd.read_parquet(ROOT / "data" / "processed" / "cleaned_master.parquet")
dev, cal, test = temporal_split(df)

adv_names = ["RandomForest", "ExtraTrees", "HistGradientBoosting", "XGBoost", "CatBoost"]
records = []

for name in adv_names:
    t0 = time.time()
    model = create_model(name, quick=True, random_state=42)
    model.fit(dev[FEATURES_A], dev[TARGETS_A].values)
    train_sec = round(time.time() - t0, 3)
    
    preds = model.predict(cal[FEATURES_A])
    m_list = regression_metrics(cal[TARGETS_A].values, preds, TARGETS_A)
    for m in m_list:
        records.append({"model": name, "train_time_sec": train_sec, **m})

df_adv = pd.DataFrame(records)
print(df_adv[["model", "target", "RMSE", "MAE", "R2", "train_time_sec"]].to_string(index=False))

out_adv = ROOT / "experiments" / "advanced_model_metrics.csv"
df_adv.to_csv(out_adv, index=False)
print(f"Saved advanced model metrics to {out_adv.relative_to(ROOT)}")
print("Advanced model evaluation COMPLETE.")


               model target    RMSE    MAE      R2  train_time_sec
        RandomForest     CO  1.4304 0.7516  0.4919           0.873
        RandomForest    NOX 10.3846 9.0021 -0.0847           0.873
          ExtraTrees     CO  1.3290 0.7187  0.5614           0.264
          ExtraTrees    NOX 10.6042 9.2416 -0.1310           0.264
HistGradientBoosting     CO  1.2620 0.7283  0.6045          23.204
HistGradientBoosting    NOX 10.6522 9.3462 -0.1413          23.204
             XGBoost     CO  1.3918 0.7971  0.5190           4.941
             XGBoost    NOX 10.5598 9.3439 -0.1216           4.941
            CatBoost     CO  1.2540 0.6567  0.6095           2.893
            CatBoost    NOX 10.5188 9.2910 -0.1129           2.893
Saved advanced model metrics to experiments\advanced_model_metrics.csv
Advanced model evaluation COMPLETE.
